## 设计手册

### 整体结构
1. 存储模块
增 删 改 查
指定保存文件根目录

包括每个ticker的 独立的内容 以及可能的元信息
以及全局的内容, 例如市场所有的ticker的list 之类的 全局信息

    1. 数据库基本上只有读取 和 完全的写入这两个情况 不存在部分修改, (即获取新的df, 然后顶替掉旧的 
    2. 我需要保存的内容就是bundle的结构 
    3. 我希望尽可能可以单独的更新bundle的内容, 例如, 假如我单独获得了 data 中的 financial_indicator_snapshot, 我可以单独的更新这部分内容 我希望有一个函数, 首先检查是否拥有想要单独修改的这个key financial_indicator_snapshot, 如果有的话, 进行替换, (方便对一些有时效性的数据进行更新) 
    4. 它能够保存df这个数据结构, 方便, 简单的读取, 恢复

2.  指定一个dict 用于关联表的内容 以及 文件名称, 可以考虑带上中文->英文之间的翻译的表
起到翻译各种结构的作用

3. 指定要爬取的内容, 给出一些name 与 lambda 函数 的dict
将函数的ticker传入, 就可以获得爬取的df

4. 爬取, 一个框架, 传入 上面的dict name->lambda, 然后获得爬取的信息
最后写入数据库

5. 数据分析 TODO, 这个是后面需要考虑的任务, 现在只需要把数据爬下来




hk-quant-strategy-platform/
  __init__.py

  models/
    bundle.py              # Bundle / DatasetResult 的数据结构（纯数据，不做IO）

  registry/
    fetchers.py            # fetcher 注册表（name -> callable）
    datasets.py            # dataset 的元信息：文件名、中文/英文名、类别、是否快照等
    schema.py              # 字段/科目映射（中文->英文、同义词、标准列名）

  fetch/
    akshare_hk.py          # 具体用 akshare 抓的实现（你的 lambda 最终会落在这里）

  storage/
    base.py                # StorageBackend 抽象接口（save/load/list/delete）
    parquet_store.py       # ParquetStore：你现在最适配的实现

  services/
    crawler.py             # 爬取框架：给 ticker + registry -> bundle
    writer.py              # bundle -> store（全量覆盖写）
    reader.py              # 高频读取API：按需加载、缓存、只读快取

  utils/
    time.py                # asof、时间戳格式

### 需要保存的内容

akshare 中的
1. 元信息与公司画像
    1. 证券信息
    2. 是否港股通标的
    3. 所属行业
    4. 公司介绍/主营描述
    5. stock_hk_company_profile_em（行业 + 公司介绍等）
    6. stock_hk_security_profile_em（是否港股通、板块等）
2. 财务报表
    1. stock_financial_hk_report_em(stock=..., symbol=资产负债表/利润表/现金流量表, indicator="年度")
3. 分红与回报
    1. 每次分红的：公告日、除净日、派息日、派息额、股权登记日等 stock_hk_fhpx_detail_ths(symbol=...)
    2. *注销型回购* akshare 好像没有直接的注销型回购, 忽略这个应该并不影响太大, 自己看的时候主动考虑即可
4. 市值 股本
    1. 总市值(港元)
    2. 已发行股本
    3. stock_hk_financial_indicator_em(symbol=...)
5. 行情与流动性
    1. 成交量、成交额
    2. 换手率
    3. 涨跌幅
    4. stock_hk_spot_em()（全市场, 只保存全部市场的数据一次即可, 每一个数据结构中, 不需要保存这个数据）



### 存储的数据结构
一个dict
1. ticker
2. asof
3. data

每个data 中, 又是一个dict, 保存着上面每一次调用akshare获得的df, 用一个独特的dict key/name 访问


### 财务报表内容分析

#### 资产负债表
不同公司的年报中科目的行数不同, 大致在40左右, 可能是由于披露的细粒度不同or有某些行业有独特的科目的原因

其大致按照港股年报中展示的顺序排列, 
1. 非流动资产（固定资产、无形资产、商誉、递延税项资产…）
2. 非流动资产合计
3. 流动资产（存货、应收、短投、现金…）
4. 流动资产合计
5. 总资产
6. 流动负债（应付、税项、短贷、合同负债、租赁负债流动部分…）
7. 流动负债合计
8. 一些派生项（净流动资产、总资产减流动负债）
9. 非流动负债（长期贷款、递延税项负债、租赁负债非流动部分…）
10. 非流动负债合计
11. 总负债
12. 少数股东权益（如有）
13. 股东权益明细（股本、储备、保留溢利、库存股、OCI…）
14. 总权益/净资产
15. 平衡项（总权益及总负债 = 总资产）

df中有一个`STD_ITEM_CODE`列, 给不同的科目分配了一个独特的id, 是比科目更加合适的key
每一个科目唯一对应一个 STD_ITEM_CODE, 应该使用STD_ITEM_CODE编辑公式, 获取数据

当获取完所有的数据的时候, 检查数据的一致性, 返回STD_ITEM_CODE的列表, 查看是否真的唯一
1. 有些公司的code一致, 有些不一致: 下面的猜想已经得到大概的验证
    1. 银行是001 开头
    2. 保险是002 开头
    3. 证券是003 开头
    4. 其他大部分是004 开头

#### 利润表
安排的逻辑和资产负债表差不多
`STD_ITEM_CODE`编号与资产负债表重合, 即同一个编号, 在不同的表中有不同的意思
大部分是遵循
1. 银行是001 开头 
2. 保险是002 开头
3. 证券是003 开头
4. 其他大部分是004 开头
1 2 3中对应类别的公司占比80%-90%以上
1 2 3 开头的很少, 大部分都是004开头的一般企业

#### 现金流量表
`STD_ITEM_CODE` 不同, 用了更短的code, 应该是所有的报表统一使用同一个code


#### ETF
港股中还有大量的ETF, 例如07711
应该剔除这些ETF, 这些不能作为股票分析, 哪里可以标注这些是否是股票

现在我发现, 一个判定的指标:
非公司标的会拥有 `fetcher_dict["company_profile"](ticker)`, 但是大部分与公司有关的东西都是None, 例如注册地, 行业, 董事长, 公司介绍等等
同时基本上(我当前看见的是所有)所有的非公司的标的都没有
`fetcher_dict["security_profile"](ticker)`
`fetcher_dict["financial_indicator_snapshot"](ticker)`
可以使用这个机制, 进行股票的判断.

我们重新修改了 fetcher中的 hk_spot_all, 让其每次更新的时候, 同步判断是否是股票, 然后给df增加一列
这样的改动最小, 最持久

香港交易所有一个 https://www.hkex.com.hk/chi/services/trading/securities/securitieslists/ListOfSecurities_c.xlsx
列出了所有上市的公司


### 还要做的任务 TODO
添加 市盈率 市净率 等指标的分析, 以及增长情况的分析




In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
from typing import Callable, Dict, Any, Optional, Union

import akshare as ak


In [2]:
# 将项目根目录 (stock) 加入 path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)
# 再修改了库文件之后 自动重新import
%load_ext autoreload
%autoreload 2

---

### 初始化

In [3]:
from lib.hk_quant_strategy_platform.crawler import crawl_ticker
from lib.hk_quant_strategy_platform.storage import ParquetStore
from lib.hk_quant_strategy_platform.fetchers import build_ticker_fetchers, hk_spot_all
from lib.hk_quant_strategy_platform.refresh import (
    refresh_market,
    refresh_global_only,
    refresh_financial_indicator_snapshot_only,
)


In [4]:
storage_dir = r"E:\Program\Python\stock\data\stock_data"

In [5]:
storage = ParquetStore(storage_dir)
fetcher_dict = build_ticker_fetchers(indicator="年度")

In [6]:
# 初始化全局的股票列表 执行的频率很低, 一年一次都可以
# df_spot = refresh_global_only(storage, hk_spot_all_fetcher=hk_spot_all)

In [7]:
# 直接从内存中读出初始化的股票列表
df_spot = storage.load_global_df("hk_spot_all")

In [8]:
df_spot

,序号,代码,名称,最新价,涨跌额,涨跌幅,今开,最高,最低,昨收,成交量,成交额,is_equity
0,1,00399,星太链集团,0.135,0.059,77.63,0.076,0.145,0.069,0.076,97905000.0,11478025.0,True
1,2,08620,亚洲速运,0.149,0.053,55.21,0.127,0.158,0.125,0.096,102080000.0,8224580.0,True
2,3,02427,GUANZE MEDICAL,2.990,1.040,53.33,2.010,2.990,2.010,1.950,9465000.0,23830550.0,True
3,4,01129,中国水业集团,0.450,0.150,50.00,0.320,0.495,0.320,0.300,3793600.0,1573812.0,True
4,5,08283,中食民安,0.640,0.155,31.96,0.500,0.660,0.500,0.485,2202812.0,1315020.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4619,4620,00036,远东控股国际,NaN,NaN,NaN,NaN,NaN,NaN,0.275,NaN,NaN,True
4620,4621,00033,INTL GENIUS,NaN,NaN,NaN,NaN,NaN,NaN,0.400,NaN,NaN,True
4621,4622,00011,恒生银行,NaN,NaN,NaN,NaN,NaN,NaN,154.300,NaN,NaN,False
4622,4623,00009,金奥国际,NaN,NaN,NaN,NaN,NaN,NaN,0.013,NaN,NaN,False


---

### 分析股票判别列表的差异

分析我这里获得的股票列表(明确是股本类型, 有财报 资产负债表之类的)
和 从香港交易所拿到的官方的表格之间的差异

差异已经写到下面的地方了 E:\Program\Python\stock\data\stock_data\globals\stock_ticker_diff.txt


In [9]:
# 分析
# -----------------------------
# 0) 配置：HKEX 列表 CSV 路径
# -----------------------------
hkex_csv = r"E:\Program\Python\stock\data\stock_data\globals\ListOfSecurities_c.csv"

In [10]:

# -----------------------------
# 1) 规范化工具
# -----------------------------
def norm_ticker(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip().replace("\u3000", "").replace("\xa0", "")
    return s.zfill(5) if s.isdigit() else s


def norm_name(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip().replace("\u3000", " ").replace("\xa0", " ")
    return " ".join(s.split())


def build_pairs_from_df_spot(df_spot: pd.DataFrame, *, only_equity: bool = True) -> pd.DataFrame:
    df = df_spot.copy()

    if only_equity:
        # is_equity 可能是 boolean dtype，含 <NA>；这里把 NA 当 False
        df = df[df["is_equity"].fillna(False)]

    df["ticker"] = df["代码"].map(norm_ticker)
    df["name"] = df["名称"].map(norm_name)

    df = df[df["ticker"] != ""]
    # 同一 ticker 在 spot 里通常不会重复，但稳妥起见去重
    return df[["ticker", "name"]].drop_duplicates("ticker", keep="first")


def load_hkex_pairs(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, encoding="utf-8-sig", dtype={"股份代號": str})
    df["ticker"] = df["股份代號"].map(norm_ticker)
    df["name"] = df["股份名稱"].map(norm_name)
    df = df[df["ticker"] != ""]
    return df[["ticker", "name"]].drop_duplicates("ticker", keep="first")

In [11]:

# -----------------------------
# 2) 构造两个 (ticker, name) 表
# -----------------------------
pairs_spot_equity = build_pairs_from_df_spot(df_spot, only_equity=True)
pairs_hkex = load_hkex_pairs(hkex_csv)

In [12]:
# -----------------------------
# 3) ticker 差集
# -----------------------------
set_spot = set(pairs_spot_equity["ticker"])
set_hkex = set(pairs_hkex["ticker"])

only_in_spot = sorted(set_spot - set_hkex)
only_in_hkex = sorted(set_hkex - set_spot)
in_both = sorted(set_spot & set_hkex)

print(f"spot(is_equity=True) tickers: {len(set_spot)}")
print(f"HKEX tickers              : {len(set_hkex)}")
print(f"Only in spot              : {len(only_in_spot)}")
print(f"Only in HKEX              : {len(only_in_hkex)}")
print(f"In both                   : {len(in_both)}")

# 只在 spot 的：带名称
only_in_spot_df = pairs_spot_equity[pairs_spot_equity["ticker"].isin(only_in_spot)].sort_values("ticker")
only_in_hkex_df = pairs_hkex[pairs_hkex["ticker"].isin(only_in_hkex)].sort_values("ticker")

print("\n--- Only in spot (sample) ---")
print(only_in_spot_df.head(30).to_string(index=False))

print("\n--- Only in HKEX (sample) ---")
print(only_in_hkex_df.head(30).to_string(index=False))



spot(is_equity=True) tickers: 2751
HKEX tickers              : 2753
Only in spot              : 6
Only in HKEX              : 8
In both                   : 2745

--- Only in spot (sample) ---
ticker   name
 02553   首钢朗泽
 02615   百利天恒
 02912 均安控股股权
 02913 恒益控股股权
 03588 富途控股-W
 07489   岚图汽车

--- Only in HKEX (sample) ---
ticker            name
 02908        巨濤海洋石油股權
 04332         AMGEN-T
 04333            思科－Ｔ
 04335           英特爾－Ｔ
 04336          應用材料－Ｔ
 04337           星巴克－Ｔ
 04338            微軟－Ｔ
 04621 CINDA 21USDPREF


In [13]:
# # 按照上面的建议, 重新修改df_spot, 增加REITS和巨濤海洋石油股權
# # 不过 巨濤海洋石油股權 根本没有, 数据库没有这一行, 根本无法添加进入
# 
# # 你手工要强制当作股票的 ticker
# manual_true = [
#     "00405","00435","00778","00808","00823","01426",
#     "01503","01881","02191","02778","02908","87001",
# ]
# 
# # 1) 规范化代码列，确保是 5 位字符串
# df_spot["代码"] = df_spot["代码"].astype(str).str.zfill(5)
# 
# # 2) 确保 is_equity 列存在且是 boolean dtype（可选，但建议）
# if "is_equity" not in df_spot.columns:
#     df_spot["is_equity"] = pd.Series([pd.NA] * len(df_spot), dtype="boolean")
# else:
#     df_spot["is_equity"] = df_spot["is_equity"].astype("boolean")
# 
# # 3) 强制置 True
# df_spot.loc[df_spot["代码"].isin(manual_true), "is_equity"] = True
# 
# # 4) 覆盖写回全局 parquet（原子写）
# storage.save_global_df("hk_spot_all", df_spot, overwrite=True)
# 
# # 5) 可选：立刻读回校验
# df_check = storage.load_global_df("hk_spot_all")
# print(df_check[df_check["代码"].isin(manual_true)][["代码","名称","is_equity"]])


---

### 获取所有ticker的详细数据

按照上面的搜集到的ticker的列表, 逐个爬取公司的内容, 将其保存下来

In [14]:
# spot_df, result = refresh_market(
#     storage,
#     fetcher_dict,
#     mode="tickers_only",
#     tickers=None,          # 使用已保存的 hk_spot_all.parquet -> is_equity=True 股票池
#     datasets=None,         # None = 抓取所有 fetchers 注册的数据集
#     incremental=False,     # 首次初始化：全部强制抓
#     prune=False,           # 首次建议 False（等稳定后再考虑 full refresh prune=True）
#     sleep_s=0.5,           # 限速，避免被上游限流；你可按实际调大/调小
#     progress=True,
# )


---

### 检查 ticker 数据的完整性


检查数据质量, 是否出现了df因为网络之类的, 没有保存的现象

In [15]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Set, Tuple
import pandas as pd


@dataclass(slots=True)
class AuditConfig:
    required_keys: Sequence[str]
    allow_empty_keys: Set[str] = None  # 允许 empty 的 key（例如 dividend_history）
    must_nonempty_keys: Set[str] = None  # 必须非空的 key（例如 company_profile, financial_indicator_snapshot）
    check_all_na: bool = False  # 是否读取 parquet 判断 all_na
    check_parquet_exists: bool = True  # missing 是否同时检查 parquet 文件存在（更严谨）


def audit_store_quality(
        store,
        *,
        required_keys: Sequence[str],
        allow_empty_keys: Optional[Set[str]] = None,
        must_nonempty_keys: Optional[Set[str]] = None,
        check_all_na: bool = False,
        check_parquet_exists: bool = True,
        tickers: Optional[Sequence[str]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Enhanced audit:
    - dataset summary: ok/empty/all_na/error/missing/total + rates
    - per ticker detail: lists missing/error/empty/all_na/suspicious_empty

    Definitions (per ticker, per dataset key):
      - missing: manifest.datasets does not include key OR (check_parquet_exists and file not found)
      - error  : manifest.errors includes key (even if df exists)
      - empty  : rows==0 in manifest.datasets[key]["rows"]
      - all_na : df exists and rows>0, but all values are NA (requires check_all_na=True)
      - ok     : present, not error, and (rows>0 or key in allow_empty_keys)

    suspicious_empty:
      - rows==0 but key in must_nonempty_keys
      - OR all_na for key in must_nonempty_keys
    """
    allow_empty_keys = allow_empty_keys or set()
    must_nonempty_keys = must_nonempty_keys or set()

    if tickers is None:
        tickers = store.list_tickers()

    # Prepare accumulators
    per_dataset = {k: {"ok": 0, "empty": 0, "all_na": 0, "error": 0, "missing": 0, "total": 0} for k in required_keys}

    rows_detail = []

    for t in tickers:
        m = store.get_manifest(t)
        ds_meta: Dict[str, dict] = (m.get("datasets") or {})
        ds_err: Dict[str, str] = (m.get("errors") or {})

        missing_keys = []
        error_keys = []
        empty_keys = []
        all_na_keys = []
        suspicious_empty_keys = []

        for k in required_keys:
            per_dataset[k]["total"] += 1

            # 1) missing?
            meta = ds_meta.get(k)
            if meta is None:
                per_dataset[k]["missing"] += 1
                missing_keys.append(k)
                continue

            # optional: check parquet exists on disk
            if check_parquet_exists:
                # use store.has_dataset which checks manifest+file
                if not store.has_dataset(t, k):
                    per_dataset[k]["missing"] += 1
                    missing_keys.append(k)
                    continue

            # 2) error?
            if k in ds_err:
                per_dataset[k]["error"] += 1
                error_keys.append(k)
                # 注意：error 和 empty/all_na 可以并存，但通常你希望 error 优先级最高
                # 这里仍然继续判断 rows==0，便于你看错误是不是“正常空被当 error”
                # 不 continue

            # 3) empty? (manifest rows)
            rows = int(meta.get("rows", 0) or 0)
            if rows == 0:
                per_dataset[k]["empty"] += 1
                empty_keys.append(k)
                if k in must_nonempty_keys:
                    suspicious_empty_keys.append(k)
                # empty 情况下不需要 all_na 检测
                # ok 的判断：empty 但允许空 => ok；否则不算 ok
                if (k not in ds_err) and (k in allow_empty_keys):
                    per_dataset[k]["ok"] += 1
                continue

            # 4) all_na? (needs reading parquet)
            if check_all_na:
                try:
                    b = store.load_bundle(t, load=[k])
                    df = b.data.get(k)
                    if df is None:
                        # treat as missing
                        per_dataset[k]["missing"] += 1
                        missing_keys.append(k)
                        continue
                    if df.isna().all(axis=None):
                        per_dataset[k]["all_na"] += 1
                        all_na_keys.append(k)
                        if k in must_nonempty_keys:
                            suspicious_empty_keys.append(k)
                        # all_na 不算 ok（除非你明确允许）
                        continue
                except Exception:
                    # load failure -> treat as missing (or you can treat as error)
                    per_dataset[k]["missing"] += 1
                    missing_keys.append(k)
                    continue

            # 5) ok
            if k not in ds_err:
                per_dataset[k]["ok"] += 1
            else:
                # error already counted; do not count ok
                pass

        rows_detail.append(
            {
                "ticker": t,
                "missing_n": len(missing_keys),
                "error_n": len(error_keys),
                "empty_n": len(empty_keys),
                "all_na_n": len(all_na_keys),
                "suspicious_empty_n": len(suspicious_empty_keys),
                "missing_keys": ",".join(missing_keys),
                "error_keys": ",".join(error_keys),
                "empty_keys": ",".join(empty_keys),
                "all_na_keys": ",".join(all_na_keys),
                "suspicious_empty_keys": ",".join(suspicious_empty_keys),
            }
        )

    df_ticker = pd.DataFrame(rows_detail).sort_values(
        ["suspicious_empty_n", "error_n", "missing_n", "empty_n", "all_na_n"],
        ascending=False,
    )

    # Build dataset summary df
    dataset_rows = []
    for k in required_keys:
        d = per_dataset[k]
        total = d["total"] or 1
        dataset_rows.append(
            {
                "dataset": k,
                "ok": d["ok"],
                "empty": d["empty"],
                "all_na": d["all_na"],
                "missing": d["missing"],
                "error": d["error"],
                "total": d["total"],
                "ok_rate": d["ok"] / total,
                "empty_rate": d["empty"] / total,
                "all_na_rate": d["all_na"] / total,
                "missing_rate": d["missing"] / total,
                "error_rate": d["error"] / total,
            }
        )

    df_dataset = pd.DataFrame(dataset_rows).sort_values(
        ["error_rate", "missing_rate", "empty_rate", "all_na_rate"],
        ascending=False,
    )

    return df_dataset.reset_index(drop=True), df_ticker.reset_index(drop=True)

In [16]:
required_keys = list(fetcher_dict.keys())
allow_empty_keys = {}
must_nonempty_keys = {"company_profile", "financial_indicator_snapshot"}

In [17]:
# df_dataset, df_ticker = audit_store_quality(
#     storage,
#     required_keys=required_keys,
#     allow_empty_keys=allow_empty_keys,
#     must_nonempty_keys=must_nonempty_keys,
#     check_all_na=True,               # True会访问数据库,检查是不是空的df, False则只检查error的返回结果
#     check_parquet_exists=True,        # 更严谨，文件不存在算 missing
# )
# df_dataset, df_ticker


结论: 只有11个REITS什么都没有, 剩下有123个公司没有历史的分红数据. 根据代码, 我们可以保证没有真的就是空的df表格, 这个没有被忽略

下面再重新执行一下这123个股票的分红的部分, 如果还是123, 就能基本上保证他们就是没有

In [18]:
import random
from collections import Counter

from lib.hk_quant_strategy_platform.refresh import refresh_market
from lib.hk_quant_strategy_platform.utils import normalize_hk_ticker

DATASET = "dividend_history"

# 你提到的 11 个港股 REIT（可选剔除）
REITS = {
    "00405", "00435", "00778", "00808", "00823",
    "01426", "01503", "01881", "02191", "02778", "87001"
}


def get_failed_tickers_for_dataset(store, dataset_key: str) -> list[str]:
    failed = []
    for t in store.list_tickers():  # 已经是 5 位
        m = store.get_manifest(t)
        if dataset_key in (m.get("errors") or {}):
            failed.append(t)
    return failed


def error_reason_counter(store, dataset_key: str, tickers: list[str]) -> Counter:
    cnt = Counter()
    for t in tickers:
        m = store.get_manifest(t)
        msg = (m.get("errors") or {}).get(dataset_key)
        if msg:
            cnt[msg] += 1
    return cnt

In [19]:
# # 1) 获取失败列表
# failed_all = get_failed_tickers_for_dataset(storage, DATASET)
# failed_no_reits = [t for t in failed_all if normalize_hk_ticker(t) not in REITS]
# 
# print(f"[retry] {DATASET} failed tickers: {len(failed_all)}")
# print(f"[retry] {DATASET} failed tickers (exclude REITs): {len(failed_no_reits)}")
# 
# # （可选）先看失败原因 TopN，判断是否主要是 “No tables found”
# cnt0 = error_reason_counter(storage, DATASET, failed_no_reits)
# for msg, n in cnt0.most_common(10):
#     print(n, msg)

In [20]:
# # 2) 只重抓 dividend_history（不会动其它数据集）
# # 注意：你 refresh_market 里加了 whitelist 交集过滤；如果列表里有不在 hk_spot_all 股票池的，会被自动 drop 掉
# spot_df, res = refresh_market(
#     storage,
#     fetcher_dict,
#     mode="tickers_only",
#     tickers=failed_no_reits,
#     datasets=[DATASET],                # 只更新股息
#     incremental=False,                 # 强制重抓
#     strategy="update_datasets",        # 安全：不覆盖 manifest 的其他 datasets
#     prune=False,
#     update_asof_on_update_datasets=True,
#     skip_errors=True,
#     sleep_s=0.7,                       # THS 容易限流，建议 >= 0.5
#     progress=True,
# )

然后再回头, 跑之前检查缺失数量的程序, 如果一致就能保证数据是完整的, 可以进入下一步的分析了.

---

### 分析数据的结构

例如 分析财报的各个科目的id的分散情况, 例如和行业的关系, 

#### 简单输出公司的摘要

In [21]:
import pandas as pd
from typing import Iterable, Optional, Any, Dict


def build_company_overview_df(storage, tickers: Optional[Iterable[str]] = None) -> pd.DataFrame:
    """
    生成“公司基本信息增强表”（一行一个ticker），数据来源：
      - company_profile
      - security_profile
      - financial_indicator_snapshot

    输出列（按你的要求）：
      ticker, 名称, 注册地, 成立日期, 行业, 员工人数, 公司介绍, 港股通, 市值(港元), 股息率TTM(%), PE, PB
    """
    if tickers is None:
        tickers = storage.list_tickers()

    def _first_value(df: Optional[pd.DataFrame], candidates: Iterable[str]) -> Any:
        if df is None or not isinstance(df, pd.DataFrame) or df.empty:
            return None
        for c in candidates:
            if c in df.columns:
                v = df.iloc[0][c]
                if pd.isna(v):
                    return None
                return v
        return None

    def _as_bool(v: Any) -> Optional[bool]:
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return None
        if isinstance(v, bool):
            return v
        s = str(v).strip().lower()
        if s in {"是", "y", "yes", "true", "1"}:
            return True
        if s in {"否", "n", "no", "false", "0"}:
            return False
        return None

    rows = []
    for t in tickers:
        try:
            b = storage.load_bundle(t)
        except Exception:
            # 读不到bundle就跳过（也可改成记录错误行）
            continue

        cp = b.data.get("company_profile")
        sp = b.data.get("security_profile")
        fi = b.data.get("financial_indicator_snapshot")

        name = _first_value(cp, ["公司名称", "名称"]) or _first_value(sp, ["证券简称", "简称", "name"])
        reg_place = _first_value(cp, ["注册地"])
        founded = _first_value(cp, ["公司成立日期", "成立日期"])
        industry = _first_value(cp, ["所属行业", "行业"])
        employees = _first_value(cp, ["员工人数", "雇员人数", "员工总数"])
        intro = _first_value(cp, ["公司介绍", "公司简介", "介绍", "简介"])

        sh = _as_bool(_first_value(sp, ["是否沪港通标的", "沪港通标的", "是否沪港通"]))
        sz = _as_bool(_first_value(sp, ["是否深港通标的", "深港通标的", "是否深港通"]))
        hk_connect = None if (sh is None and sz is None) else bool(sh or sz)

        mktcap = _first_value(fi, ["总市值(港元)", "总市值", "市值(港元)", "市值"])
        div_yield = _first_value(fi, ["股息率TTM(%)", "股息率TTM", "股息率(%)", "股息率"])
        pe = _first_value(fi, ["市盈率", "PE", "PE(TTM)", "市盈率(TTM)"])
        pb = _first_value(fi, ["市净率", "PB", "市净率(PB)"])

        rows.append(
            {
                "ticker": b.ticker,  # bundle里通常已是 5位 "00001"
                "名称": name,
                "注册地": reg_place,
                "成立日期": founded,
                "行业": industry,
                "员工人数": employees,
                "公司介绍": intro,
                "港股通": hk_connect,  # 沪港通/深港通 OR 后的一列
                "市值(港元)": mktcap,
                "股息率TTM(%)": div_yield,
                "PE": pe,
                "PB": pb,
            }
        )

    df = pd.DataFrame(rows)

    # 可选：把数值列转成 numeric（失败则保持原样）
    for col in ["员工人数", "市值(港元)", "股息率TTM(%)", "PE", "PB"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "ticker" in df.columns:
        df = df.sort_values("ticker").reset_index(drop=True)

    return df

In [22]:
# 用法示例：
# df_overview = build_company_overview_df(storage)  # 默认用 storage.list_tickers()

In [23]:
# storage.save_global_df("company_overview_df", df_overview, overwrite=True)

In [24]:
df_overview = storage.load_global_df("company_overview_df")

In [25]:
df_overview

,ticker,名称,注册地,成立日期,行业,员工人数,公司介绍,港股通,市值(港元),股息率TTM(%),PE,PB
0,00001,长江和记实业有限公司,Cayman Islands 开曼群岛（英属）,2014-12-11,综合企业,173817.0,"长江和记实业有限公司(长江和记实业)是业务遍布全球的大型跨国综合企业,一向锐意创新,...",True,2.435908e+11,3.496855,31.492027,0.439069
1,00002,中电控股有限公司,中国香港特别行政区,1901-01-25,公用事业,8442.0,"中电控股有限公司在香港经营纵向式的发电、输电和配电服务以及零售业务,致力为香港八成人...",True,1.932735e+11,4.117647,16.931535,1.826954
2,00003,香港中华煤气有限公司,中国香港特别行政区,1862,公用事业,2155.0,"香港中华煤气有限公司(煤气公司)于1862年成立,是香港历史最悠久的公用事业机构,亦...",True,1.446140e+11,4.516129,25.662164,2.510154
3,00004,九龙仓集团有限公司,中国香港特别行政区,1886-11-15,地产,6000.0,"九龙仓集团有限公司(‘九龙仓’,股份代号:0004)始创于1886年,历史悠久,是香...",True,7.896775e+10,1.547988,-1518.610502,0.551051
4,00005,汇丰控股有限公司,United Kingdom 英国,1959-01-01,银行,211130.0,"汇丰控股有限公司设于伦敦,汇丰集团是全球最大规模的银行及金融服务机构之一,在欧洲、亚...",True,2.304917e+12,3.822935,17.888144,1.547077
...,...,...,...,...,...,...,...,...,...,...,...,...
2746,86618,京东健康股份有限公司,Cayman Islands 开曼群岛（英属）,2018-11-30,药品及生物科技,4572.0,"京东健康股份有限公司是中国最大的在线医疗健康平台,也是医疗产业链数字化改造的领跑者,...",False,1.712148e+11,NaN,36.265822,2.953388
2747,87001,汇贤房托管理有限公司,None,2010-10-26,None,902.0,None,None,NaN,NaN,NaN,NaN
2748,89618,京东集团股份有限公司,Cayman Islands 开曼群岛（英属）,2006-11-06,专业零售,570895.0,京东集团股份有限公司京东(JD.COM)是中国领先的技术驱动型电商和零售基础设施服务...,False,2.970800e+11,3.846137,9.226660,1.280887
2749,89888,百度集团股份有限公司,Cayman Islands 开曼群岛（英属）,2000-01-18,媒体及娱乐,35900.0,"百度集团股份有限公司成立于2000年,是一家拥有强大互联网基础的领先人工智能和互联网...",False,3.319452e+11,NaN,36.886896,1.254858


#### 分析数据中财报中的id和名称的关系


结论如下
每个id唯一对应一个中文名称
但是中文名称可能会对应多个id, 不过每个id都是不同的00x开头(001,002,003,004), 代表不同的行业, 实际上也不能算作重复

资产负债表和利润表由于有00x的行业的表示, 存在这个现象

但是现金流量表没有00x的行业, 完全是一一对应

In [26]:
import pandas as pd
from typing import Iterable, Optional, Any, List


def _first_value(df: Optional[pd.DataFrame], candidates: Iterable[str]) -> Any:
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return None
    for c in candidates:
        if c in df.columns:
            v = df.iloc[0][c]
            if pd.isna(v):
                return None
            return v
    return None


def build_fin_item_fact_table(
        storage,
        *,
        dataset_key: str,  # e.g. "financial_bs_yearly" / "financial_is_yearly" / "financial_cf_yearly"
        tickers: Optional[Iterable[str]] = None,
        add_dataset_col: bool = True,
        require_stock_like: bool = False,  # 过滤ETF/非公司标的：要求 security_profile + financial_indicator_snapshot 都存在
) -> pd.DataFrame:
    """
    通用：从任意一个财务明细表(dataset_key)抽取“科目事实表”：
      ticker, 公司名称(可选), 行业(可选), report_year, STD_ITEM_CODE, STD_ITEM_NAME
    """
    if tickers is None:
        tickers = storage.list_tickers()

    parts: List[pd.DataFrame] = []

    for t in tickers:
        # 没有目标dataset直接跳过
        if not storage.has_dataset(t, dataset_key):
            continue

        # 可选：过滤非股票标的（你之前的经验规则）
        if require_stock_like:
            if not storage.has_dataset(t, "security_profile"):
                continue
            if not storage.has_dataset(t, "financial_indicator_snapshot"):
                continue

        # 只加载需要的dataset，避免整包读取
        load_keys = [dataset_key, "company_profile", "security_profile"]
        b = storage.load_bundle(t, load=load_keys)

        df = b.data.get(dataset_key)
        if df is None or df.empty:
            continue

        # 取公司名称/行业（可选）
        cp = b.data.get("company_profile")
        sp = b.data.get("security_profile")
        company_name = _first_value(cp, ["公司名称"]) or _first_value(sp, ["证券简称", "SECURITY_NAME_ABBR"])
        industry = _first_value(cp, ["所属行业", "行业"])

        # 年份：优先 REPORT_DATE（你的样例三张都有）
        if "REPORT_DATE" not in df.columns:
            # 若极少数表缺失，可改成 fallback 到 STD_REPORT_DATE
            if "STD_REPORT_DATE" in df.columns:
                date_col = "STD_REPORT_DATE"
            else:
                continue
        else:
            date_col = "REPORT_DATE"

        report_year = pd.to_datetime(df[date_col], errors="coerce").dt.year

        sub = df.loc[:, ["STD_ITEM_CODE", "STD_ITEM_NAME"]].copy()
        sub["report_year"] = report_year
        sub["ticker"] = b.ticker
        sub["公司名称"] = company_name
        sub["行业"] = industry
        if add_dataset_col:
            sub["dataset"] = dataset_key

        # 清洗 + 去重（同公司同年同科目保留一条）
        sub = sub.dropna(subset=["STD_ITEM_CODE", "STD_ITEM_NAME", "report_year"])
        sub["STD_ITEM_CODE"] = sub["STD_ITEM_CODE"].astype(str).str.strip()
        sub["STD_ITEM_NAME"] = sub["STD_ITEM_NAME"].astype(str).str.strip()
        sub["report_year"] = sub["report_year"].astype(int)

        dedup_cols = ["ticker", "report_year", "STD_ITEM_CODE", "STD_ITEM_NAME"]
        if add_dataset_col:
            dedup_cols.append("dataset")

        sub = sub.drop_duplicates(subset=dedup_cols)
        parts.append(sub)

    if not parts:
        cols = ["ticker", "公司名称", "行业", "report_year", "STD_ITEM_CODE", "STD_ITEM_NAME"]
        if add_dataset_col:
            cols.append("dataset")
        return pd.DataFrame(columns=cols)

    out = pd.concat(parts, ignore_index=True)
    # 可选排序，便于检查
    out = out.sort_values(["ticker", "report_year", "STD_ITEM_CODE"]).reset_index(drop=True)
    return out


In [27]:
df_bs = build_fin_item_fact_table(storage, dataset_key="financial_bs_yearly", require_stock_like=True)
df_is = build_fin_item_fact_table(storage, dataset_key="financial_is_yearly", require_stock_like=True)
df_cf = build_fin_item_fact_table(storage, dataset_key="financial_cf_yearly", require_stock_like=True)

In [28]:
# 这段代码没什么用了, 这个就是下面这个两个方向映射表格的退化版本 build_item_code_mappings

import pandas as pd
from typing import Dict, Any, Optional


def check_code_name_bijection(
        df: pd.DataFrame,
        *,
        code_col: str = "STD_ITEM_CODE",
        name_col: str = "STD_ITEM_NAME",
        normalize: bool = True,
) -> Dict[str, Any]:
    """
    检查 code <-> name 是否一一对应（双射/ bijection）。

    返回：
      {
        "ok_bijection": bool,                 # True 表示 code->name 唯一 且 name->code 唯一
        "n_pairs": int,                       # (code,name) 唯一对数量
        "n_codes": int,
        "n_names": int,
        "code_to_name_conflicts": DataFrame,  # code 对应多个 name 的冲突列表
        "name_to_code_conflicts": DataFrame,  # name 对应多个 code 的冲突列表
      }
    """
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return {
            "ok_bijection": True,
            "n_pairs": 0,
            "n_codes": 0,
            "n_names": 0,
            "code_to_name_conflicts": pd.DataFrame(columns=[code_col, "name_count", "names"]),
            "name_to_code_conflicts": pd.DataFrame(columns=[name_col, "code_count", "codes"]),
        }

    if code_col not in df.columns or name_col not in df.columns:
        raise KeyError(f"Missing columns: need {code_col} and {name_col}")

    x = df[[code_col, name_col]].copy()

    # 规范化：去空格、把 NaN/空串干掉
    if normalize:
        x[code_col] = x[code_col].astype(str).str.strip()
        x[name_col] = x[name_col].astype(str).str.strip()
        # 把 'nan'/'None' 这种字符串也当成缺失
        x = x.replace({code_col: {"nan": None, "None": None, "": None},
                       name_col: {"nan": None, "None": None, "": None}})
    x = x.dropna(subset=[code_col, name_col])

    # 去重到唯一 (code,name) 对
    pairs = x.drop_duplicates(subset=[code_col, name_col])

    # code -> name 冲突
    code_conf = (
        pairs.groupby(code_col)[name_col]
        .agg(name_count="nunique", names=lambda s: sorted(set(s)))
        .reset_index()
        .query("name_count > 1")
        .sort_values(["name_count", code_col], ascending=[False, True])
        .reset_index(drop=True)
    )

    # name -> code 冲突
    name_conf = (
        pairs.groupby(name_col)[code_col]
        .agg(code_count="nunique", codes=lambda s: sorted(set(s)))
        .reset_index()
        .query("code_count > 1")
        .sort_values(["code_count", name_col], ascending=[False, True])
        .reset_index(drop=True)
    )

    ok = (len(code_conf) == 0) and (len(name_conf) == 0)

    return {
        "ok_bijection": ok,
        "n_pairs": int(len(pairs)),
        "n_codes": int(pairs[code_col].nunique()),
        "n_names": int(pairs[name_col].nunique()),
        "code_to_name_conflicts": code_conf,
        "name_to_code_conflicts": name_conf,
    }


In [29]:
# r_bs = check_code_name_bijection(df_bs, code_col="STD_ITEM_CODE", name_col="STD_ITEM_NAME")
# r_bs["name_to_code_conflicts"]
# r_is = check_code_name_bijection(df_is, code_col="STD_ITEM_CODE", name_col="STD_ITEM_NAME")
# r_is["name_to_code_conflicts"]
# r_cf = check_code_name_bijection(df_cf, code_col="STD_ITEM_CODE", name_col="STD_ITEM_NAME")
# r_cf["name_to_code_conflicts"]


输出财务报表的科目和id的映射的df, 用于编写后续的筛选股票的算法, 指标

使用同一个中文科目可能有不同的id的名称

同时也可以详细显示有多少个科目

正反输出两个表格: 中文->id(一对多), id->中文(一一对应)

In [30]:
import pandas as pd
from typing import Tuple, List, Any, Dict


def build_item_code_mappings(
        df_xx: pd.DataFrame,
        *,
        code_col: str = "STD_ITEM_CODE",
        name_col: str = "STD_ITEM_NAME",
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    """
    输入：df_xx（至少包含 STD_ITEM_CODE / STD_ITEM_NAME）
    输出（更“干净”的两张映射表）：

    1) df_name_to_codes（中文 -> id，一对多，允许同名多code）
       列：
         - STD_ITEM_NAME
         - code_count        # 对应多少个不同 code
         - codes            # code 列表（排序去重）

    2) df_code_to_names（id -> 中文，通常一对一；如有冲突则一对多）
       列：
         - STD_ITEM_CODE
         - name_count       # 对应多少个不同中文名
         - names            # 中文名列表（排序去重）

       说明：这里不再额外给 STD_ITEM_NAME 单列，因为你说冲突时会被 names 覆盖语义。
       若你确实需要“一对一”版本，可在外部用 name_count==1 筛并取 names[0]。

    3) stats：总体统计信息（便于你打印/断言/监控）
    """
    # ---------- empty / validation ----------
    if df_xx is None or not isinstance(df_xx, pd.DataFrame) or df_xx.empty:
        df_name_to_codes = pd.DataFrame(columns=[name_col, "code_count", "codes"])
        df_code_to_names = pd.DataFrame(columns=[code_col, "name_count", "names"])
        stats = {
            "rows": 0,
            "n_pairs": 0,
            "n_codes": 0,
            "n_names": 0,
            "ok_code_to_name_unique": True,
            "ok_name_to_code_unique": True,
            "n_code_to_name_conflicts": 0,
            "n_name_to_code_multi": 0,
        }
        return df_name_to_codes, df_code_to_names, stats

    if code_col not in df_xx.columns or name_col not in df_xx.columns:
        raise KeyError(f"df_xx 必须包含列：{code_col}, {name_col}")

    # ---------- normalize ----------
    x = df_xx[[code_col, name_col]].copy()
    x[code_col] = x[code_col].astype(str).str.strip()
    x[name_col] = x[name_col].astype(str).str.strip()

    # 将常见“伪缺失”归一为 None
    x = x.replace(
        {
            code_col: {"": None, "nan": None, "None": None, "NaN": None},
            name_col: {"": None, "nan": None, "None": None, "NaN": None},
        }
    ).dropna(subset=[code_col, name_col])

    # 唯一 (code, name) 对
    pairs = x.drop_duplicates(subset=[code_col, name_col]).copy()

    def _sorted_list(s: pd.Series) -> List[str]:
        return sorted(set(str(v) for v in s.dropna().tolist()))

    # ---------- 1) name -> codes ----------
    df_name_to_codes = (
        pairs.groupby(name_col)[code_col]
        .agg(code_count="nunique", codes=_sorted_list)
        .reset_index()
        .sort_values(["code_count", name_col], ascending=[False, True])
        .reset_index(drop=True)
    )

    # ---------- 2) code -> names ----------
    df_code_to_names = (
        pairs.groupby(code_col)[name_col]
        .agg(name_count="nunique", names=_sorted_list)
        .reset_index()
        .sort_values(["name_count", code_col], ascending=[False, True])  # 冲突的放前面
        .reset_index(drop=True)
    )

    # ---------- stats ----------
    stats = {
        "rows": int(len(df_xx)),
        "n_pairs": int(len(pairs)),
        "n_codes": int(pairs[code_col].nunique()),
        "n_names": int(pairs[name_col].nunique()),
        "ok_code_to_name_unique": bool((df_code_to_names["name_count"] == 1).all()),
        "ok_name_to_code_unique": bool((df_name_to_codes["code_count"] == 1).all()),
        "n_code_to_name_conflicts": int((df_code_to_names["name_count"] > 1).sum()),
        "n_name_to_code_multi": int((df_name_to_codes["code_count"] > 1).sum()),
    }

    return df_name_to_codes, df_code_to_names, stats


# -----------------------
# 可选：从 df_code_to_names 导出“一对一版本”（仅保留不冲突的code）
# -----------------------
def code_to_name_one_to_one(
        df_code_to_names: pd.DataFrame,
        *,
        code_col: str = "STD_ITEM_CODE",
        names_col: str = "names",
) -> pd.DataFrame:
    """
    输入：df_code_to_names（含 name_count/names）
    输出：仅保留 name_count==1 的一对一映射：
      STD_ITEM_CODE, STD_ITEM_NAME
    """
    if df_code_to_names is None or df_code_to_names.empty:
        return pd.DataFrame(columns=[code_col, "STD_ITEM_NAME"])

    ok = df_code_to_names[df_code_to_names["name_count"] == 1].copy()
    ok["STD_ITEM_NAME"] = ok[names_col].apply(lambda xs: xs[0] if isinstance(xs, list) and xs else None)
    return ok[[code_col, "STD_ITEM_NAME"]].reset_index(drop=True)

In [31]:
# -----------------------
# 用法示例
# -----------------------
# bs_name2code, bs_code2name, bs_stats = build_item_code_mappings(df_bs)
# is_name2code, is_code2name, is_stats = build_item_code_mappings(df_is)
# cf_name2code, cf_code2name, cf_stats = build_item_code_mappings(df_cf)


In [32]:
# 保存成文件
import json
from pathlib import Path
import pandas as pd
from typing import Tuple


def save_mapping_dfs(
        out_dir: str,
        *,
        bs_name2code: pd.DataFrame, bs_code2name: pd.DataFrame,
        is_name2code: pd.DataFrame, is_code2name: pd.DataFrame,
        cf_name2code: pd.DataFrame, cf_code2name: pd.DataFrame,
        also_csv: bool = False,  # 额外导出 CSV 方便肉眼看
) -> None:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    files = {
        "bs_name2code": bs_name2code,
        "bs_code2name": bs_code2name,
        "is_name2code": is_name2code,
        "is_code2name": is_code2name,
        "cf_name2code": cf_name2code,
        "cf_code2name": cf_code2name,
    }

    # jsonl：list 原生可保存，恢复无损
    for name, df in files.items():
        path = out / f"{name}.jsonl"
        df.to_json(path, orient="records", lines=True, force_ascii=False)

    if also_csv:
        # CSV：更适合直接打开看；list 会变字符串，但人类可读
        for name, df in files.items():
            path = out / f"{name}.csv"
            df.to_csv(path, index=False, encoding="utf-8-sig")  # Windows Excel 友好


def load_mapping_dfs(out_dir: str) -> Tuple[
    pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    out = Path(out_dir)

    def _read(name: str) -> pd.DataFrame:
        path = out / f"{name}.jsonl"
        return pd.read_json(path, orient="records", lines=True)

    bs_name2code = _read("bs_name2code")
    bs_code2name = _read("bs_code2name")
    is_name2code = _read("is_name2code")
    is_code2name = _read("is_code2name")
    cf_name2code = _read("cf_name2code")
    cf_code2name = _read("cf_code2name")
    return bs_name2code, bs_code2name, is_name2code, is_code2name, cf_name2code, cf_code2name


In [33]:
financial_statement_mapping_table_dir = r"E:\Program\Python\stock\data\stock_data"

In [34]:
# # 写入, 方便未来不需要处理就可以直接读取
# # 其实 还可以直接写入数据库的global字段, 作为df直接保存, 但是这样就不是人类可读的了
# save_mapping_dfs(financial_statement_mapping_table_dir, bs_name2code=bs_name2code, bs_code2name=bs_code2name,
#                  is_name2code=is_name2code, is_code2name=is_code2name,
#                  cf_name2code=cf_name2code, cf_code2name=cf_code2name,
#                  also_csv=False)

In [35]:
bs_name2code, bs_code2name, is_name2code, is_code2name, cf_name2code, cf_code2name = load_mapping_dfs(
    financial_statement_mapping_table_dir)

In [36]:
bs_name2code

,STD_ITEM_NAME,code_count,codes
0,交易性金融负债,4,"[001002006, 002002006, 003015011, 004020011]"
1,保留溢利(累计亏损),4,"[001009004, 002009004, 003025004, 004030004]"
2,储备,4,"[001009002, 002009002, 003025002, 004030002]"
3,公积金,4,"[001009007, 002009007, 003025005, 004030005]"
4,其他储备,4,"[001009006, 002009006, 003025009, 004030009]"
...,...,...,...
217,附属公司及其他权益,1,[001001009]
218,非流动资产平衡项目,1,[004001998]
219,预收保费,1,[002002016]
220,香港特区政府流通纸币,1,[001002003]


In [37]:
is_name2code

,STD_ITEM_NAME,code_count,codes
0,全面收益其他项目,4,"[001040997, 002040997, 003040997, 004040997]"
1,全面收益总额,4,"[001039999, 002039999, 003039999, 004039999]"
2,其他全面收益,4,"[001030999, 002030999, 003030999, 004030999]"
3,其他全面收益其他项目,4,"[001030997, 002030997, 003030997, 004030997]"
4,利息收入,4,"[001001001, 002003004, 003001002, 004011200]"
...,...,...,...
104,财务费用,1,[002011003]
105,赔付支出,1,[002007001]
106,銷售及分銷費用,1,[003007008]
107,销售及分销费用,1,[004010003]


In [64]:
cf_name2code

,STD_ITEM_NAME,code_count,codes
0,偿还借款,1,[007002]
1,偿还融资租赁,1,[007011]
2,公司及附属公司之EBITDA,1,[001013]
3,减:出售资产之溢利,1,[001008]
4,减:利息收入,1,[001002]
...,...,...,...
77,递延收入(增加)减少,1,[002009]
78,除税前溢利(业务利润),1,[001001]
79,非运算项目,1,[013999]
80,预付款项、按金及其他应收款项减少(增加),1,[002007]


In [39]:
def _parse_codes(x) -> list[str]:
    """
    把 codes 列（可能是字符串形式的 list，也可能已经是 list）解析成 list[str]，
    并保留前导 0（统一补齐到 9 位：如 4020011 -> '004020011'）。
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    codes = x if isinstance(x, list) else ast.literal_eval(str(x))
    out = []
    for c in codes:
        s = str(c).strip()
        # 典型码是 9 位；如果更长就不截断
        out.append(s.zfill(9) if len(s) < 9 else s)
    return out

def _sort_codes_list(codes: list[str]) -> list[str]:
    """
    codes 内部排序：004 优先，其次 001/002/003；各前缀内按字典序升序（等价于数值升序）。
    """
    rank = {"004": 0, "001": 1, "002": 2, "003": 3}
    return sorted(codes, key=lambda s: (rank.get(s[:3], 99), s))

def _pick_prefix(codes: list[str], prefix: str) -> str:
    """从 codes 里挑出某个前缀的码；没有则给一个很大的哨兵值，保证排序时排在后面。"""
    for c in codes:
        if c.startswith(prefix):
            return c
    return "999999999"

def sort_bs_name2code(bs_name2code: pd.DataFrame) -> pd.DataFrame:
    """
    行排序规则：
      1) 先按 004 前缀的 code 升序（如 004001001 ...）
      2) 再按 001、002、003 的 code 依次升序
    同时会把每行 codes 列内部也按 004 -> 001 -> 002 -> 003 的顺序整理。
    """
    df = bs_name2code.copy()

    df["_codes_list"] = df["codes"].apply(_parse_codes).apply(_sort_codes_list)
    df["_004"] = df["_codes_list"].apply(lambda lst: _pick_prefix(lst, "004"))
    df["_001"] = df["_codes_list"].apply(lambda lst: _pick_prefix(lst, "001"))
    df["_002"] = df["_codes_list"].apply(lambda lst: _pick_prefix(lst, "002"))
    df["_003"] = df["_codes_list"].apply(lambda lst: _pick_prefix(lst, "003"))

    # mergesort 稳定排序，便于可预测
    df = df.sort_values(["_004", "_001", "_002", "_003"], ascending=True, kind="mergesort") \
           .reset_index(drop=True)

    # 如果希望 codes 仍然是字符串形式的 list（和你示例一致）
    df["codes"] = df["_codes_list"].apply(lambda lst: str(lst))

    return df.drop(columns=["_codes_list", "_004", "_001", "_002", "_003"])

In [40]:
bs_name2code_sorted = sort_bs_name2code(bs_name2code)


In [41]:
bs_name2code_sorted

,STD_ITEM_NAME,code_count,codes
0,固定资产,3,"['004001001', '001001014', '003001001']"
1,物业厂房及设备,2,"['004001002', '003001002']"
2,投资物业,4,"['004001003', '001001012', '002001025', '00300..."
3,无形资产,4,"['004001004', '001001015', '002001004', '00300..."
4,商誉,4,"['004001005', '001001016', '002001003', '00300..."
...,...,...,...
217,代理买卖证券款,1,['003007020']
218,可转换票据及债券(流动),1,['003007022']
219,总资产减总负债,1,['003012001']
220,长期银行贷款,1,['003015001']


#### 按照行业分析开头的编号00x

In [42]:
import pandas as pd
from typing import Optional, List, Any


def analyze_prefix_industry(
        df_fact: pd.DataFrame,
        *,
        statement: str,
        ticker_col: str = "ticker",
        name_col: str = "公司名称",
        industry_col: str = "行业",
        code_col: str = "STD_ITEM_CODE",
        prefix_len: int = 3,
        include_tickers: bool = True,
        include_names: bool = True,
        max_list: int = 50,  # 每行最多列举多少个 ticker / 名称，避免单元格过大
) -> pd.DataFrame:
    """
    单表分析：prefix = STD_ITEM_CODE[:3] 的行业分布（按“覆盖公司数”口径）

    输出列（示例）：
      statement, prefix, 行业, n_tickers, total_tickers, pct, tickers(list), names(list)

    说明：
      - 统计口径：对每个 (prefix, 行业) 统计唯一 ticker 数
      - pct = n_tickers / total_tickers_in_prefix
      - total_tickers_in_prefix：该 prefix 覆盖的唯一 ticker 总数
    """
    if df_fact is None or not isinstance(df_fact, pd.DataFrame) or df_fact.empty:
        cols = ["statement", "prefix", industry_col, "n_tickers", "total_tickers", "pct"]
        if include_tickers:
            cols.append("tickers")
        if include_names:
            cols.append("names")
        return pd.DataFrame(columns=cols)

    # 必要列检查
    for c in [ticker_col, industry_col]:
        if c not in df_fact.columns:
            raise KeyError(f"df_fact 缺少必要列：{c}")
    if code_col not in df_fact.columns and "prefix3" not in df_fact.columns:
        raise KeyError(f"df_fact 需要 {code_col} 或 prefix3 其中之一用于生成 prefix")

    x = df_fact.copy()

    # prefix
    if "prefix3" in x.columns:
        x["prefix"] = x["prefix3"].astype(str).str.strip().str[:prefix_len]
    else:
        x["prefix"] = x[code_col].astype(str).str.strip().str[:prefix_len]

    # industry normalize
    x[industry_col] = x[industry_col].fillna("UNKNOWN").astype(str).str.strip()

    # 只保留必要列（可选 name）
    keep_cols = ["prefix", ticker_col, industry_col]
    if include_names and name_col in x.columns:
        keep_cols.append(name_col)
    x = x[keep_cols].copy()

    # drop bad
    x = x.replace({"prefix": {"": None, "nan": None, "None": None}})
    x = x.dropna(subset=["prefix", ticker_col])

    # 去重到 (ticker, prefix, 行业) —— 避免年份/科目行数影响
    keys = ["prefix", ticker_col, industry_col]
    x_u = x.drop_duplicates(subset=keys)

    # 每个 prefix 的总覆盖 ticker 数
    prefix_totals = (
        x_u.groupby("prefix")[ticker_col]
        .nunique()
        .reset_index(name="total_tickers")
    )

    # (prefix, 行业) 覆盖 ticker 数
    grp = (
        x_u.groupby(["prefix", industry_col])[ticker_col]
        .nunique()
        .reset_index(name="n_tickers")
    )

    out = grp.merge(prefix_totals, on="prefix", how="left")
    out["pct"] = out["n_tickers"] / out["total_tickers"]
    out.insert(0, "statement", statement)

    # 可选：附带 ticker / 名称列表（用于你手工 spot check）
    if include_tickers or (include_names and name_col in x_u.columns):
        # 聚合列表用 set 去重，再截断
        def _list_unique(s: pd.Series) -> List[Any]:
            uniq = pd.unique(s.dropna())
            # 排序 ticker 更直观；名称不一定需要排序
            try:
                uniq_sorted = sorted(uniq)
            except Exception:
                uniq_sorted = list(uniq)
            return uniq_sorted[:max_list]

        if include_tickers:
            tickers_df = (
                x_u.groupby(["prefix", industry_col])[ticker_col]
                .apply(_list_unique)
                .reset_index(name="tickers")
            )
            out = out.merge(tickers_df, on=["prefix", industry_col], how="left")

        if include_names and name_col in x_u.columns:
            names_df = (
                x_u.groupby(["prefix", industry_col])[name_col]
                .apply(_list_unique)
                .reset_index(name="names")
            )
            out = out.merge(names_df, on=["prefix", industry_col], how="left")

    # 排序：prefix 内按 n_tickers 降序
    out = out.sort_values(["prefix", "n_tickers"], ascending=[True, False]).reset_index(drop=True)

    # 列顺序对齐你示例
    cols = ["statement", "prefix", industry_col, "n_tickers", "total_tickers", "pct"]
    if include_tickers:
        cols.append("tickers")
    if include_names and ("names" in out.columns):
        cols.append("names")
    out = out[cols]

    return out

In [43]:
# # 这两个表格完全一样
# bs_pi = analyze_prefix_industry(df_bs, statement="BS", include_tickers=True, include_names=True, max_list=100)
# is_pi = analyze_prefix_industry(df_is, statement="IS", include_tickers=True, include_names=True, max_list=100)
# bs_pi

#### 分析企业行业分布

In [44]:
import pandas as pd
from typing import List, Any


def summarize_overview_by_industry(
        df_overview: pd.DataFrame,
        *,
        industry_col: str = "行业",
        ticker_col: str = "ticker",
        name_col: str = "名称",
        max_list: int = 200,
) -> pd.DataFrame:
    if df_overview is None or not isinstance(df_overview, pd.DataFrame) or df_overview.empty:
        return pd.DataFrame(columns=[industry_col, "n_companies", "tickers", "company_names"])

    for c in [industry_col, ticker_col]:
        if c not in df_overview.columns:
            raise KeyError(f"df_overview 缺少必要列：{c}")

    x = df_overview.copy()
    x[industry_col] = x[industry_col].fillna("UNKNOWN").astype(str).str.strip()
    x[ticker_col] = x[ticker_col].astype(str).str.strip()

    keep_cols = [industry_col, ticker_col]
    if name_col in x.columns:
        keep_cols.append(name_col)
    x = x[keep_cols].drop_duplicates(subset=[ticker_col])  # ticker 级去重

    def _sorted_unique(s: pd.Series) -> List[Any]:
        vals = sorted(set(str(v) for v in s.dropna().tolist()))
        return vals[:max_list]

    def _unique_names(s: pd.Series) -> List[str]:
        # 去重保持稳定顺序：按出现顺序取前 max_list 个
        out, seen = [], set()
        for v in s.dropna().astype(str).str.strip().tolist():
            if v and v not in seen:
                out.append(v);
                seen.add(v)
            if len(out) >= max_list:
                break
        return out

    agg_dict = {
        "n_companies": (ticker_col, "nunique"),
        "tickers": (ticker_col, _sorted_unique),
    }
    if name_col in x.columns:
        agg_dict["company_names"] = (name_col, _unique_names)
    else:
        # 没有公司名称列就返回空 list
        x["__dummy_name__"] = None
        agg_dict["company_names"] = ("__dummy_name__", lambda s: [])

    df_industry = (
        x.groupby(industry_col, dropna=False)
        .agg(**agg_dict)
        .reset_index()
        .sort_values(["n_companies", industry_col], ascending=[False, True])
        .reset_index(drop=True)
    )
    return df_industry

In [45]:
df_industry_summary = summarize_overview_by_industry(df_overview, max_list=200)

In [46]:
df_industry_summary

,行业,n_companies,tickers,company_names
0,地产,272,"[00004, 00010, 00012, 00014, 00016, 00017, 000...","[九龙仓集团有限公司, 恒隆集团有限公司, 恒基兆业地产有限公司, 希慎兴业有限公司, 新鸿..."
1,建筑,222,"[00022, 00058, 00240, 00331, 00366, 00368, 003...","[茂盛控股有限公司, 新威国际控股有限公司, 利基控股有限公司, 丰盛生活服务有限公司, 陆..."
2,工业工程,215,"[00031, 00038, 00042, 00057, 00076, 00096, 000...","[中国航天国际控股有限公司, 第一拖拉机股份有限公司, 东北电气发展股份有限公司, 震雄集团..."
3,其他金融,168,"[00064, 00080, 00086, 00093, 00111, 00129, 001...","[结好控股有限公司, CAI控股, 新鸿基有限公司, 零在科技金融集团有限公司, 信达国际控..."
4,药品及生物科技,164,"[00013, 00239, 00241, 00314, 00348, 00455, 004...","[和黄医药(中国)有限公司, 白花油国际有限公司, 阿里健康信息技术有限公司, 思派健康科技..."
5,软件服务,160,"[00020, 00046, 00100, 00248, 00268, 00302, 003...","[商汤集团股份有限公司, 科联系统集团有限公司, MiniMax Group Inc., 香..."
6,旅游及消闲设施,132,"[00027, 00037, 00045, 00051, 00052, 00062, 000...","[银河娱乐集团有限公司, 远东酒店实业有限公司, 香港上海大酒店有限公司, 海港企业有限公司..."
7,其他医疗保健,127,"[00030, 00157, 00178, 00286, 00379, 00383, 004...","[云白国际有限公司, 自然美生物科技有限公司, 莎莎国际控股有限公司, 爱帝宫母婴健康股份有..."
8,纺织及服饰,118,"[00113, 00125, 00130, 00146, 00210, 00256, 002...","[迪生创建(国际)有限公司, 新兴光学集团控股有限公司, 慕诗国际集团有限公司, 太平地毡国..."
9,媒体及娱乐,116,"[00018, 00072, 00082, 00136, 00150, 00164, 002...","[东方企控集团有限公司, 超媒体控股有限公司, 疯狂体育集团有限公司, 中国儒意控股有限公司..."


### 使用外挂的公司的分析指标筛选公司

In [47]:
import pandas as pd
import re
import json
from typing import Any, List, Dict, Tuple


def _parse_codes_cell(x: Any) -> List[str]:
    """
    把 codes 单元格解析成纯数字字符串列表：
      "[001002006, 002002006, 003015011]" -> ["001002006","002002006","003015011"]
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    s = str(x)
    # 抽取所有数字串
    codes = re.findall(r"\d+", s)
    # 去重、保持顺序
    seen = set()
    out = []
    for c in codes:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out


def filter_items_without_004(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, List[str]]]:
    """
    筛选出：codes 列表中完全不包含以 '004' 开头的 code 的科目。
    返回：
      - df_non004_only: DataFrame（带解析后的 codes_list / prefixes）
      - name2codes_non004_only: {STD_ITEM_NAME: [codes...]}
    """
    required = {"STD_ITEM_NAME", "codes"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"df missing columns: {missing}")

    dfx = df.copy()
    dfx["codes_list"] = dfx["codes"].apply(_parse_codes_cell)

    def has_004(codes: List[str]) -> bool:
        return any(str(c).startswith("004") for c in (codes or []))

    dfx["has_004"] = dfx["codes_list"].apply(has_004)
    dfx["prefixes"] = dfx["codes_list"].apply(lambda cs: sorted({str(c)[:3] for c in cs if len(str(c)) >= 3}))

    df_non004_only = dfx[~dfx["has_004"]].copy()

    # name -> codes_list（去重合并）
    name2codes: Dict[str, List[str]] = {}
    for _, r in df_non004_only.iterrows():
        name = str(r["STD_ITEM_NAME"]).strip()
        codes = r["codes_list"] or []
        if not name:
            continue
        prev = name2codes.get(name, [])
        merged = sorted(set(prev).union(codes))
        name2codes[name] = merged

    return df_non004_only, name2codes


def print_name2codes(name2codes: Dict[str, List[str]]) -> None:
    print(json.dumps(name2codes, ensure_ascii=False, indent=2))


In [48]:
# ---------------------------
# 用法：
df_non004_only, name2codes_non004_only = filter_items_without_004(bs_name2code)

In [49]:
df_non004_only

,STD_ITEM_NAME,code_count,codes,codes_list,has_004,prefixes
33,可收回税项,3,"[001001027, 002001021, 003002021]","[001001027, 002001021, 003002021]",False,"[001, 002, 003]"
37,归属于母公司股东权益,3,"[001009999, 002009999, 003025999]","[001009999, 002009999, 003025999]",False,"[001, 002, 003]"
38,归属于母公司股东权益其他项目,3,"[001009997, 002009997, 003025997]","[001009997, 002009997, 003025997]",False,"[001, 002, 003]"
40,股东权益合计,3,"[001011999, 002011999, 003029999]","[001011999, 002011999, 003029999]",False,"[001, 002, 003]"
45,非控股股东权益,3,"[001003001, 002003001, 003021001]","[001003001, 002003001, 003021001]",False,"[001, 002, 003]"
...,...,...,...,...,...,...
216,长期银行贷款,1,[003015001],[003015001],False,[003]
217,附属公司及其他权益,1,[001001009],[001001009],False,[001]
219,预收保费,1,[002002016],[002002016],False,[002]
220,香港特区政府流通纸币,1,[001002003],[001002003],False,[001]


In [67]:
from lib.hk_quant_strategy_platform.analyzer_function import run_analyzer, compute_weighted_from_bundle, analyzer_basic_info_balance_sheet, get_report_df
from lib.hk_quant_strategy_platform.analyzer_parameter import *

In [68]:
bundle = storage.load_bundle("0001")
total_asset = compute_weighted_from_bundle(bundle, "financial_bs_yearly", offset=0, meta_blacklist=["codes"], 
                                      weights=DEFAULT_ASSET_WEIGHTS, return_detail=False)
total_asset

{'CashOnlyAsset': 112331430120.0,
 'ConservativeAsset': 182136880944.0,
 'NormalAsset': 262532803020.0}

In [52]:
total_liability = compute_weighted_from_bundle(bundle, "financial_bs_yearly", offset=0, meta_blacklist=["codes"], 
                                      weights=DEFAULT_LIAB_WEIGHTS, return_detail=False)

In [53]:
total_liability

{'PriorityLiab': 330740779281.6, 'TotalLiab': 425932098000.0}

In [54]:
df = run_analyzer(
    storage,
    lambda b: analyzer_basic_info_balance_sheet(
        b,
        bs_key="financial_bs_yearly",
        bs_offset=0,
        asset_weights=DEFAULT_ASSET_WEIGHTS,
        liab_weights=DEFAULT_LIAB_WEIGHTS
    ),
    # tickers = ["0001", "0005"],
    keep_skipped=True,
)

In [55]:
df

,ticker,asof,ok,error,skipped,名称,注册地,成立日期,行业,员工人数,...,Liab_Total,MOS_CashOnlyAsset_minus_TotalLiab,MOS_CashOnlyAsset_minus_PriorityLiab,MOS_ConservativeAsset_minus_TotalLiab,MOS_ConservativeAsset_minus_PriorityLiab,MOS_NormalAsset_minus_TotalLiab,MOS_NormalAsset_minus_PriorityLiab,bs_offset,LR_CashOnlyAsset_over_PriorityLiab,LR_NormalAsset_over_TotalLiab
0,00001,2026-02-19 20:19:46.842548+00:00,True,None,False,长江和记实业有限公司,Cayman Islands 开曼群岛（英属）,2014-12-11,综合企业,173817.0,...,4.259321e+11,-1.287408,-0.896624,-1.000839,-0.610055,-0.670794,-0.280010,0,0.339636,0.616372
1,00002,2026-02-19 20:50:20.524966+00:00,True,None,False,中电控股有限公司,中国香港特别行政区,1901-01-25,公用事业,8442.0,...,1.144539e+11,-0.566840,-0.360806,-0.454761,-0.248727,-0.334074,-0.128040,0,0.065639,0.435864
2,00003,2026-02-19 20:07:49.003567+00:00,True,None,False,香港中华煤气有限公司,中国香港特别行政区,1862,公用事业,2155.0,...,8.328350e+10,-0.534798,-0.387580,-0.410199,-0.262981,-0.278935,-0.131717,0,0.095885,0.515656
3,00004,2026-02-19 20:59:32.396903+00:00,True,None,False,九龙仓集团有限公司,中国香港特别行政区,1886-11-15,地产,6000.0,...,4.405080e+10,-0.454168,-0.177472,-0.224492,0.052204,-0.027766,0.248930,0,0.368735,0.950225
4,00005,2026-02-19 20:54:16.020789+00:00,True,None,False,汇丰控股有限公司,United Kingdom 英国,1959-01-01,银行,211130.0,...,2.030561e+13,-6.878374,-4.909119,-5.922231,-3.952976,-4.805405,-2.836150,0,0.282339,0.454532
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2746,86618,2026-02-20 16:52:39.034486+00:00,True,None,False,京东健康股份有限公司,Cayman Islands 开曼群岛（英属）,2018-11-30,药品及生物科技,4572.0,...,1.603416e+10,0.157420,0.217538,0.174224,0.234342,0.204214,0.264332,0,7.487498,3.180629
2747,87001,2026-02-19 20:46:43.751212+00:00,True,None,False,汇贤房托管理有限公司,None,2010-10-26,None,902.0,...,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN
2748,89618,2026-02-20 16:52:39.948853+00:00,True,None,False,京东集团股份有限公司,Cayman Islands 开曼群岛（英属）,2006-11-06,专业零售,570895.0,...,3.849370e+11,-0.621881,0.197925,-0.462367,0.357439,-0.291448,0.528358,0,1.415871,0.775071
2749,89888,2026-02-20 16:52:40.897110+00:00,True,None,False,百度集团股份有限公司,Cayman Islands 开曼群岛（英属）,2000-01-18,媒体及娱乐,35900.0,...,1.441680e+11,0.082280,0.304078,0.213015,0.434812,0.330734,0.552531,0,2.430849,1.761512


In [58]:
def screen_bs_mos(
    df: pd.DataFrame,
    *,
    cap_min: float = 1e8,                      # 市值下限：50亿港元
    mos_min: float = -0.5,                      # MOS_NormalAsset_minus_PriorityLiab 下限
    require_hk_connect: bool = False,           # 港股通
    min_asset_to_debt: float = 0.4,            
) -> pd.DataFrame:

    d = df.copy()

    # 1) 数据质量闸门
    mask = (d["ok"] == True) & (d["skipped"] == False) & (d["error"].isna())

    # 2) 市值下限
    mask &= d["市值(港元)"].notna() & (d["市值(港元)"] >= cap_min)

    # 3) MOS 下限
    mask &= d["MOS_NormalAsset_minus_PriorityLiab"].notna() & (d["MOS_NormalAsset_minus_PriorityLiab"] >= mos_min)

    # 4) 港股通
    if require_hk_connect and "港股通" in d.columns:
        mask &= (d["港股通"] == True)


    mask &= d["市值(港元)"].notna() & (d["LR_NormalAsset_over_TotalLiab"] >= min_asset_to_debt)

    out = d.loc[mask].sort_values(
        by=["MOS_NormalAsset_minus_PriorityLiab", "市值(港元)"],
        ascending=[False, False]
    ).reset_index(drop=True)
    return out

In [59]:
screen_bs_mos(df)

,ticker,asof,ok,error,skipped,名称,注册地,成立日期,行业,员工人数,...,Liab_Total,MOS_CashOnlyAsset_minus_TotalLiab,MOS_CashOnlyAsset_minus_PriorityLiab,MOS_ConservativeAsset_minus_TotalLiab,MOS_ConservativeAsset_minus_PriorityLiab,MOS_NormalAsset_minus_TotalLiab,MOS_NormalAsset_minus_PriorityLiab,bs_offset,LR_CashOnlyAsset_over_PriorityLiab,LR_NormalAsset_over_TotalLiab
0,01461,2026-02-19 20:39:07.542255+00:00,True,None,False,中泰期货股份有限公司,中国,2012-12-10,其他金融,740.0,...,3.120584e+10,-13.696953,9.399505,-10.223624,12.872834,-6.730468,16.365990,0,1.439255,0.848737
1,02098,2026-02-19 20:28:33.803602+00:00,True,None,False,卓尔智联集团有限公司,Cayman Islands 开曼群岛（英属）,2010-09-22,工用支援,1558.0,...,5.514180e+10,-36.248981,-12.518703,-25.857223,-2.126946,-16.134061,7.596216,0,0.282454,0.608176
2,00007,2026-02-19 21:09:00.475913+00:00,True,None,False,智富资源投资控股集团有限公司,Bermuda 百慕大,2000-05-30,家庭电器及用品,88.0,...,1.297016e+09,-11.217798,-4.699450,-3.619366,2.898981,0.853785,7.372133,0,0.155100,1.070675
3,00253,2026-02-19 20:31:36.379126+00:00,True,None,False,顺豪控股有限公司,中国香港特别行政区,1973-01-16,旅游及消闲设施,647.0,...,1.245815e+09,-5.196556,-3.775705,1.277912,2.698763,5.616530,7.037381,0,0.225693,1.891925
4,09982,2026-02-19 20:44:58.259853+00:00,True,None,False,中原建业有限公司,Cayman Islands 开曼群岛（英属）,2020-10-22,建筑,542.0,...,5.430960e+08,5.123763,6.069357,5.465167,6.410760,5.867958,6.813552,0,13.436724,5.093128
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1859,02203,2026-02-19 20:58:58.337901+00:00,True,None,False,脑洞科技有限公司,Cayman Islands 开曼群岛（英属）,2014-09-10,半导体,67.0,...,1.961436e+08,-0.904650,-0.710280,-0.801338,-0.606968,-0.683745,-0.489375,0,0.311429,0.442250
1860,01270,2026-02-19 20:00:28.754339+00:00,True,None,False,朗廷酒店投资与朗廷酒店投资有限公司,Cayman Islands 开曼群岛（英属）,2013-01-29,旅游及消闲设施,1239.0,...,6.237743e+09,-2.755232,-2.539724,-1.390531,-1.175023,-0.705108,-0.489600,0,0.047681,0.755374
1861,03931,2026-02-19 20:09:21.804311+00:00,True,None,False,中创新航科技集团股份有限公司,中国,2015-12-08,工业工程,10080.0,...,7.430661e+10,-1.301024,-0.895045,-1.147801,-0.741822,-0.896443,-0.490465,0,0.230487,0.428693
1862,08495,2026-02-19 20:33:35.389198+00:00,True,None,False,1957 & Co. (Hospitality) Limited,Cayman Islands 开曼群岛（英属）,2016-02-03,旅游及消闲设施,563.0,...,2.168730e+08,-1.309285,-0.893154,-1.116359,-0.700229,-0.910054,-0.493924,0,0.310468,0.468251


In [61]:
storage.load_bundle("0001").data.keys()

dict_keys(['company_profile', 'security_profile', 'financial_bs_yearly', 'financial_is_yearly', 'financial_cf_yearly', 'dividend_history', 'financial_indicator_snapshot'])

,SECUCODE,SECURITY_CODE,SECURITY_NAME_ABBR,ORG_CODE,REPORT_DATE,DATE_TYPE_CODE,FISCAL_YEAR,START_DATE,STD_ITEM_CODE,STD_ITEM_NAME,AMOUNT,ticker,asof,dataset
0,00001.HK,00001,长和,10434094,2024-12-31 00:00:00,001,12-31,2024-01-01 00:00:00,001001,除税前溢利(业务利润),26456962800,00001,2026-02-19T20:19:46.842548+00:00,financial_cf_yearly
1,00001.HK,00001,长和,10434094,2024-12-31 00:00:00,001,12-31,2024-01-01 00:00:00,001003,加:利息支出,12401527680,00001,2026-02-19T20:19:46.842548+00:00,financial_cf_yearly
2,00001.HK,00001,长和,10434094,2024-12-31 00:00:00,001,12-31,2024-01-01 00:00:00,001004,减:投资收益,-10657794360,00001,2026-02-19T20:19:46.842548+00:00,financial_cf_yearly
3,00001.HK,00001,长和,10434094,2024-12-31 00:00:00,001,12-31,2024-01-01 00:00:00,001005,减:应占附属公司溢利,15427826400,00001,2026-02-19T20:19:46.842548+00:00,financial_cf_yearly
4,00001.HK,00001,长和,10434094,2024-12-31 00:00:00,001,12-31,2024-01-01 00:00:00,001006,加:减值及拨备,1721508360,00001,2026-02-19T20:19:46.842548+00:00,financial_cf_yearly
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797,00001.HK,00001,长和,10434094,2000-12-31 00:00:00,001,12-31,2000-01-01 00:00:00,007999,融资业务现金净额,2540137000,00001,2026-02-19T20:19:46.842548+00:00,financial_cf_yearly
798,00001.HK,00001,长和,10434094,2000-12-31 00:00:00,001,12-31,2000-01-01 00:00:00,010999,现金净额,-963024800,00001,2026-02-19T20:19:46.842548+00:00,financial_cf_yearly
799,00001.HK,00001,长和,10434094,2000-12-31 00:00:00,001,12-31,2000-01-01 00:00:00,011001,期初现金,3539222200,00001,2026-02-19T20:19:46.842548+00:00,financial_cf_yearly
800,00001.HK,00001,长和,10434094,2000-12-31 00:00:00,001,12-31,2000-01-01 00:00:00,011999,期末现金,2576197400,00001,2026-02-19T20:19:46.842548+00:00,financial_cf_yearly


In [72]:
get_report_df(storage.load_bundle("00882").data["financial_cf_yearly"], offset = 0)

,SECUCODE,SECURITY_CODE,SECURITY_NAME_ABBR,ORG_CODE,REPORT_DATE,DATE_TYPE_CODE,FISCAL_YEAR,START_DATE,STD_ITEM_CODE,STD_ITEM_NAME,AMOUNT,ticker,asof,dataset
0,00882.HK,00882,天津发展,10009140,2024-12-31,001,12-31,2024-01-01 00:00:00,001001,除税前溢利(业务利润),8.204288e+08,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
1,00882.HK,00882,天津发展,10009140,2024-12-31,001,12-31,2024-01-01 00:00:00,001002,减:利息收入,2.249675e+08,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
2,00882.HK,00882,天津发展,10009140,2024-12-31,001,12-31,2024-01-01 00:00:00,001003,加:利息支出,1.118008e+08,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
3,00882.HK,00882,天津发展,10009140,2024-12-31,001,12-31,2024-01-01 00:00:00,001004,减:投资收益,9.929001e+06,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
4,00882.HK,00882,天津发展,10009140,2024-12-31,001,12-31,2024-01-01 00:00:00,001005,减:应占附属公司溢利,3.881737e+08,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
5,00882.HK,00882,天津发展,10009140,2024-12-31,001,12-31,2024-01-01 00:00:00,001006,加:减值及拨备,4.985892e+07,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
6,00882.HK,00882,天津发展,10009140,2024-12-31,001,12-31,2024-01-01 00:00:00,001007,减:重估盈余,-8.555684e+06,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
7,00882.HK,00882,天津发展,10009140,2024-12-31,001,12-31,2024-01-01 00:00:00,001008,减:出售资产之溢利,7.251078e+07,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
8,00882.HK,00882,天津发展,10009140,2024-12-31,001,12-31,2024-01-01 00:00:00,001009,加:折旧及摊销,1.727028e+08,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
9,00882.HK,00882,天津发展,10009140,2024-12-31,001,12-31,2024-01-01 00:00:00,001010,减:汇兑收益,-7.537040e+06,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly


In [73]:
storage.load_bundle("00882").data["financial_cf_yearly"]

,SECUCODE,SECURITY_CODE,SECURITY_NAME_ABBR,ORG_CODE,REPORT_DATE,DATE_TYPE_CODE,FISCAL_YEAR,START_DATE,STD_ITEM_CODE,STD_ITEM_NAME,AMOUNT,ticker,asof,dataset
0,00882.HK,00882,天津发展,10009140,2024-12-31 00:00:00,001,12-31,2024-01-01 00:00:00,001001,除税前溢利(业务利润),8.204288e+08,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
1,00882.HK,00882,天津发展,10009140,2024-12-31 00:00:00,001,12-31,2024-01-01 00:00:00,001002,减:利息收入,2.249675e+08,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
2,00882.HK,00882,天津发展,10009140,2024-12-31 00:00:00,001,12-31,2024-01-01 00:00:00,001003,加:利息支出,1.118008e+08,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
3,00882.HK,00882,天津发展,10009140,2024-12-31 00:00:00,001,12-31,2024-01-01 00:00:00,001004,减:投资收益,9.929001e+06,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
4,00882.HK,00882,天津发展,10009140,2024-12-31 00:00:00,001,12-31,2024-01-01 00:00:00,001005,减:应占附属公司溢利,3.881737e+08,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
874,00882.HK,00882,天津发展,10009140,2000-12-31 00:00:00,001,12-31,2000-01-01 00:00:00,010999,现金净额,5.119410e+07,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
875,00882.HK,00882,天津发展,10009140,2000-12-31 00:00:00,001,12-31,2000-01-01 00:00:00,011001,期初现金,1.227524e+09,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
876,00882.HK,00882,天津发展,10009140,2000-12-31 00:00:00,001,12-31,2000-01-01 00:00:00,011997,期间变动其他项目,4.258309e+06,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
877,00882.HK,00882,天津发展,10009140,2000-12-31 00:00:00,001,12-31,2000-01-01 00:00:00,011999,期末现金,1.282976e+09,00882,2026-02-19T20:02:35.605241+00:00,financial_cf_yearly
